In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.stats as stats

# HCI lecture, survey data analysis

First we will load up our python packages. I'll just import a typical scientific data analysis stack:

- pandas: data analysis package
- numpy: numerical computation, it's a dependency of pandas but sometimes useful to have separately imported
- seaborn: a high-level plotting package
- matplotlib: a low-level plotting package, again a dependency of seaborn, but need to have the pyplot subpack available for reasons I can't quite remember.
- scipy.stats: has statistic tests

## *let*'s load in some data

In [ ]:
# Option A (live lecture): upload the fresh PollEV export
from google.colab import files
upload = files.upload()
data_file = list(upload.keys())[0]

In [ ]:
# Option B (fallback / try at home): download the 2025 lecture's (deidentified) data
!wget -O hci_lecture_questionnaire_2025.csv https://raw.githubusercontent.com/smcclab/thirty-nine-hundred-hci/main/notebooks/hci_lecture_questionnaire_2025.csv
data_file = "hci_lecture_questionnaire_2025.csv"

In [ ]:
## the data is _messed up_ from PollEV, some code to fix it...

from io import StringIO

raw = open(data_file).read()
blocks = raw.split('\n""\n')             # questions are separated by "" lines
blocks[0] = blocks[0].split("\n", 1)[1]  # drop the report title line

frames = []
for block in blocks:
    question, table = block.split("\n", 1)  # first line is the question
    df = pd.read_csv(StringIO(table))       # the rest is a normal CSV
    df["question"] = question.strip()
    frames.append(df)

responses = pd.concat(frames, ignore_index=True)

In [ ]:
responses.head()

In [ ]:
# clean up columns:
responses = responses[["Response", "Screen name", "question"]]

# map questions to short names.
short_names = {
    "I enjoy interactive activities in lectures.":   "enjoy_interactive",
    "I usually attend lectures in person":           "attend_in_person",
    "I usually watch lectures online":               "watch_online",
    "Select the interactive activities you prefer:": "preferred_activity",
    "What kind of degree program are you in?":       "degree_program",
    "How long have you lived in Canberra?":          "time_in_canberra",
}
responses["question"] = responses["question"].replace(short_names)
responses.head()

In [ ]:
# print descriptive statistics
responses.describe()

In [ ]:
# Pivot the table from long format to wide format.
survey_data = responses.pivot(index="Screen name", columns="question", values="Response").dropna()
survey_data.head()

In [ ]:
## order the categorical columns, then convert to numbers 1-5
likert = ["Strongly Disagree", "Disagree", "Neutral",
          "Agree", "Strongly Agree"]
for col in ["enjoy_interactive", "attend_in_person", "watch_online"]:
    survey_data[col] = pd.Categorical(survey_data[col], categories=likert, ordered=True)
    survey_data[col] = survey_data[col].cat.codes + 1

survey_data.head()

## investigate the dataset

In [ ]:

print(survey_data.columns)

print("Value Counts by Category:")
print(survey_data['degree_program'].value_counts())
print(survey_data['time_in_canberra'].value_counts())

In [ ]:
# histogram
sns.set_theme(style="ticks", palette="Set2")
plt.figure(figsize=(5, 3))
sns.histplot(data=survey_data, x='enjoy_interactive', bins=5)
plt.show()

In [ ]:
# boxplots
sns.set_theme(style="ticks", palette="Set2")
plt.figure(figsize=(5, 3))
sns.boxplot(data=survey_data, x='degree_program', y='enjoy_interactive', medianprops={'linewidth': 2, 'color': 'black'})
plt.show()



In [ ]:
# more complex plots
sns.set_theme(style="ticks", palette="Set2")
plt.figure(figsize=(10, 6))
sns.boxplot(data=survey_data, x='degree_program', y='enjoy_interactive', hue='time_in_canberra', medianprops={'linewidth': 2, 'color': 'black'})
plt.show()

In [ ]:
# put this into long format for easier plotting and analysis
likert_columns = ['enjoy_interactive','attend_in_person','watch_online']
metadata_columns = ['degree_program','time_in_canberra','preferred_activity']
survey_long = pd.melt(survey_data,
                     id_vars=metadata_columns,
                     value_vars=likert_columns,
                     var_name='question',
                     value_name='score')

# survey_long['question'] = survey_long['question'].str.replace('_likert', '')
# survey_long['question'] = survey_long['question'].str.replace('_', ' ').str.title()
survey_long.head()


In [ ]:
plt.figure(figsize=(12, 7))
sns.boxplot(data=survey_long, x='question', y='score', hue='degree_program', medianprops={'linewidth': 2, 'color': 'black'})
plt.tight_layout()
plt.show()

In [ ]:
g = sns.FacetGrid(survey_long, col='time_in_canberra', height=5, aspect=1)
g.map_dataframe(sns.boxplot, x='question', y='score', hue='degree_program',
                medianprops={'linewidth': 2, 'color': 'black'})
g.add_legend(title='Degree')
g.set_axis_labels('Question', 'Likert Score')
g.set_titles('Time in CBR: {col_name}')
plt.show()

In [ ]:
g = sns.FacetGrid(survey_long, col='degree_program', height=5, aspect=1.2)
g.map_dataframe(sns.boxplot, x='question', y='score', hue='time_in_canberra',
                medianprops={'linewidth': 2, 'color': 'black'})

# Customize the plot
g.add_legend(title='time in CBR')
g.set_axis_labels('Question', 'Likert Score')
g.set_titles('{col_name}')
plt.show()

In [ ]:
undergrad_interactive = survey_data[survey_data['degree_program'] == 'Undergraduate student (Bachelor degree)']['enjoy_interactive']
postgrad_interactive = survey_data[survey_data['degree_program'] == 'Postgraduate student (Master degree)']['enjoy_interactive']

undergrad_interactive

In [ ]:
t_stat, p_value = stats.ttest_ind(undergrad_interactive, postgrad_interactive)

p_value

In [ ]:
# run some statistical tests

# is there a difference in interactive activities by degree program?

t_stat, p_value = stats.ttest_ind(undergrad_interactive, postgrad_interactive)
print(f"\nT-test: Interactive Activities by Degree Program")
print(f"T-statistic: {t_stat:.4f}")
print(f"P-value: {p_value:.4f}")

# statistic, p_val = stats.chi2_contingency(pd.crosstab(survey_data['degree_program'], survey_data['time_in_canberra']))[:2]
# print(f"\nChi-square test: Degree Program vs Time in Canberra")
# print(f"Chi-square statistic: {statistic:.4f}")
# print(f"P-value: {p_val:.4f}")
# print(f"\nCrosstab:")
# print(pd.crosstab(survey_data['degree_program'], survey_data['time_in_canberra']))